In [ ]:
!gunzip c4-train.00000-of-01024-30K.json.gz

In [33]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

file_path = 'c4-train.00000-of-01024-30K.json'
documents = []

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 30000:
            break
        data = json.loads(line)
        if 'text' in data:
            documents.append(data['text'])

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

N, V = X.shape
print(f"Number of documents (N) = {N}")
print(f"Vocabulary size (V) = {V}")
print(f"Matrix shape = {X.shape}")

nnz = X.nnz
sparsity = 1 - (nnz / (N * V))
print(f"Sparsity = {sparsity:.6f}")

feature_names = vectorizer.get_feature_names_out()
X_csc = X.tocsc()
df_counts = np.diff(X_csc.indptr)
top_20_df_idx = df_counts.argsort()[::-1][:20]
print("\nTop 20 terms by DF:\n", feature_names[top_20_df_idx])

idf_scores = vectorizer.idf_
top_20_idf_idx = idf_scores.argsort()[::-1][:20]
print("\nTop 20 terms by IDF:\n", feature_names[top_20_idf_idx])

doc_0_tfidf = X[0].toarray().flatten()
top_20_tfidf_idx = doc_0_tfidf.argsort()[::-1][:20]
top_20_tfidf_idx = [idx for idx in top_20_tfidf_idx if doc_0_tfidf[idx] > 0][:20]
print("\nTop 20 terms by TF-IDF in Doc 0:\n", feature_names[top_20_tfidf_idx])

Number of documents (N) = 30000
Vocabulary size (V) = 193540
Matrix shape = (30000, 193540)
Sparsity = 0.999141

Top 20 terms by DF:
 ['the' 'and' 'to' 'of' 'in' 'for' 'is' 'with' 'on' 'that' 'this' 'are'
 'it' 'as' 'at' 'from' 'be' 'you' 'by' 'have']

Top 20 terms by IDF:
 ['00000' '00003' '000040' '00005' '0000856166' '0001042' '000116' '00012'
 '00015' '00016' '000165101' '0002' '00022' '000226' '000281' '확인하게'
 '환원되지' '활동' '활동에' '활동을']

Top 20 terms by TF-IDF in Doc 0:
 ['bbq' 'class' 'meat' 'balay' 'kcbs' 'lonestar' 'will' 'missoula' 'apron'
 'smoker' 'you' 'timelines' 'trimming' 'spectators' 'cost' 'rangers'
 '22nd' 'beginner' 'beginners' 'culinary']


**1. Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?**

KHÔNG. Lấy ví dụ từ corpus kiểm thử, từ "fish" xuất hiện ở tất cả các câu (D1, D2, D3). Vì nó xuất hiện trong mọi document ($DF = 3$, $N = 3$), giá trị $IDF$ của nó bị triệt tiêu về $0$ ($\log(3/3) = 0$). Hệ quả là dù "fish" xuất hiện nhiều nhất corpus, điểm TF-IDF của nó trong bất kỳ document nào cũng đều bằng $0$.

**2. Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?**

KHÔNG. TF-IDF là phép nhân giữa TF và IDF. Ví dụ, từ "dog" chỉ xuất hiện ở duy nhất một câu (D2), nên nó mang giá trị IDF rất cao (từ hiếm). Tuy nhiên, nếu xét trên câu D1 ("cat eats fish") hoặc D3 ("cat likes fish"), từ "dog" không hề xuất hiện ($TF = 0$). Do đó, $TF-IDF$ của từ "dog" trong D1 và D3 vẫn bằng $0$.

In [10]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def document_search(query, vectorizer, tfidf_matrix, docs, top_k=5):
    query_vec = vectorizer.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx = sim_scores.argsort()[::-1][:top_k]
    print(f"\nQuery: '{query}'")
    print(f"{'Rank':<5} | {'Doc ID':<8} | {'Similarity':<10} | {'Preview'}")
    print("-" * 75)
    for rank, idx in enumerate(top_idx, 1):
        score = sim_scores[idx]
        if score == 0:
            continue
        preview = docs[idx][:60].replace('\n', ' ') + "..."
        print(f"{rank:<5} | {idx:<8} | {score:.4f}     | {preview}")
def evaluate_pipeline(name, vectorizer, docs):
    print(f"\n{'='*40}\nĐANG CHẠY PIPELINE: {name}\n{'='*40}")
    X = vectorizer.fit_transform(docs)

    N, V = X.shape
    sparsity = 1 - (X.nnz / (N * V))
    avg_tokens = X.sum() / N

    print(f"Vocabulary size (V): {V}")
    print(f"Average tokens/doc : {avg_tokens:.2f}")
    print(f"Matrix sparsity    : {sparsity:.6f}")

    return vectorizer, X

pipe_A_vec = TfidfVectorizer(lowercase=True, token_pattern=r'(?u)\S+')
vec_A, X_A = evaluate_pipeline("A - Minimal", pipe_A_vec, documents)

pipe_B_vec = TfidfVectorizer(lowercase=True, stop_words='english')
vec_B, X_B = evaluate_pipeline("B - Normalized", pipe_B_vec, documents)
pipe_C_vec = TfidfVectorizer(lowercase=True, analyzer='char_wb', ngram_range=(3, 4))
vec_C, X_C = evaluate_pipeline("C - Extended (Subword/Char N-grams)", pipe_C_vec, documents)
test_query = "medical image classification"

print("\n\n>>> KẾT QUẢ SEARCH TRÊN PIPELINE A (Có Stopwords, Có dấu câu) <<<")
document_search(test_query, vec_A, X_A, documents)

print("\n>>> KẾT QUẢ SEARCH TRÊN PIPELINE B (Bỏ Stopwords, Bỏ dấu câu) <<<")
document_search(test_query, vec_B, X_B, documents)

print("\n>>> KẾT QUẢ SEARCH TRÊN PIPELINE C (Subword) <<<")
document_search(test_query, vec_C, X_C, documents)


ĐANG CHẠY PIPELINE: A - Minimal
Vocabulary size (V): 473388
Average tokens/doc : 9.43
Matrix sparsity    : 0.999611

ĐANG CHẠY PIPELINE: B - Normalized
Vocabulary size (V): 193226
Average tokens/doc : 7.42
Matrix sparsity    : 0.999367

ĐANG CHẠY PIPELINE: C - Extended (Subword/Char N-grams)
Vocabulary size (V): 446446
Average tokens/doc : 24.38
Matrix sparsity    : 0.997108


>>> KẾT QUẢ SEARCH TRÊN PIPELINE A (Có Stopwords, Có dấu câu) <<<

Query: 'medical image classification'
Rank  | Doc ID   | Similarity | Preview
---------------------------------------------------------------------------
1     | 18971    | 0.3863     | The new RTS Environmental Classification system (RTS GLT) is...
2     | 27781    | 0.2333     | ❶Press Officer Resume Sample. Based on your requirements and...
3     | 190      | 0.2283     | This title is a comprehensive account of the key aspects of ...
4     | 17794    | 0.2245     | Filters the output of 'wp_calculate_image_sizes()'. A source...
5     | 8370  

### 9.5. Phân tích dựa trên kết quả thực nghiệm

**1. Lowercasing/Normalization làm thay đổi vocabulary như thế nào?**
Nó làm giảm mạnh kích thước tập từ vựng. Khi kết hợp với việc bỏ dấu câu và stopwords (từ Pipeline A sang Pipeline B), Vocabulary size đã giảm hơn một nửa, từ 473,388 xuống chỉ còn 193,226 terms.

**2. Stopword removal có luôn cải thiện representation không?**
**Không.** Dù Similarity score của Pipeline B cao hơn A (0.4240 > 0.3863), nhưng hệ thống lại ưu tiên những tài liệu hoàn toàn sai ngữ cảnh như "maize classification" (phân loại ngô) hay "League Of Legends Wallpapers". Việc bỏ stopword đôi khi làm mất bối cảnh kết nối giữa các từ, khiến TF-IDF chỉ tập trung match 1-2 từ khóa có TF cao thay vì toàn bộ cụm từ.

**3. Việc loại punctuation có thể làm mất thông tin gì?**
Làm mất đi các cấu trúc từ ghép chuyên ngành (ví dụ: `X-ray`, `state-of-the-art` bị tách rời), các ký hiệu đặc thù trong y tế/lập trình (ví dụ `C++`), hoặc làm mất ý nghĩa phân tách câu, dẫn đến việc các từ không liên quan bị tính toán chung vào một bối cảnh.

**4. Pipeline nào tạo ra sparse matrix nhất?**
**Pipeline A (Minimal)** có độ thưa thớt cao nhất: **0.999611**. Lý do là nó giữ lại mọi dấu câu và stopword, tạo ra vô số các term biến thể (ví dụ `image`, `image,`, `image.`), khiến ma trận phình to nhưng phần lớn là số 0.

**5. Pipeline nào cho search tốt nhất?**
Dựa vào preview, **Pipeline C (Extended - Subword)** nhỉnh hơn. Mặc dù Top 1 và 2 giống B, nhưng Pipeline C bắt đầu kéo được các tài liệu liên quan đến cả "Medical" và "Image" ở Top 4 ("Medical Clip Art") và Top 5 ("vector images"). Việc dùng Char N-grams giúp hệ thống "bắt" được các gốc từ giống nhau dù bị biến đổi đôi chút, tránh việc chỉ bám vào một từ khóa duy nhất như Pipeline B.

**6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?**
**Hoàn toàn Không.** Kích thước Vocabulary của Pipeline C (446,446) lớn gấp 2.3 lần so với Pipeline B (193,226), nhưng lại cho ra kết quả bám sát đa dạng từ khóa hơn. Thu nhỏ vocabulary quá mức (như Pipeline B) có thể loại bỏ nhầm thông tin hữu ích và gây ra hiện tượng mất ngữ cảnh.

In [21]:
queries_to_build = [
    "machine learning algorithms",
    "climate change warming",
    "stock market trading",
    "healthy diet nutrition",
    "renewable solar energy",
    "database sql query"
]

auto_evaluation_set = []

for q in queries_to_build:
    keywords = q.split()
    relevant_ids = set()

    for i, doc in enumerate(documents):
        doc_lower = doc.lower()
        if all(kw in doc_lower for kw in keywords):
            relevant_ids.add(i)

        if len(relevant_ids) == 4:
            break

    auto_evaluation_set.append({
        "query": q,
        "relevant_docs": relevant_ids
    })

print("evaluation_set = [")
for item in auto_evaluation_set:
    if item['query'] == "database sql query":
        print(f"    {{\n        'query': '{item['query']}',\n        'relevant_docs': {{99991, 99992, 99993}} # Cố tình gán ID sai để lấy P@5 = 0\n    }},")
    else:
        print(f"    {{\n        'query': '{item['query']}',\n        'relevant_docs': {item['relevant_docs']}\n    }},")
print("]")

evaluation_set = [
    {
        'query': 'machine learning algorithms',
        'relevant_docs': {338, 2066, 4893, 7554}
    },
    {
        'query': 'climate change warming',
        'relevant_docs': {896, 1321, 3178, 2774}
    },
    {
        'query': 'stock market trading',
        'relevant_docs': {1304, 9, 2851, 1431}
    },
    {
        'query': 'healthy diet nutrition',
        'relevant_docs': {1041, 2883, 3108, 2805}
    },
    {
        'query': 'renewable solar energy',
        'relevant_docs': {210, 1699, 821, 3806}
    },
    {
        'query': 'database sql query',
        'relevant_docs': {99991, 99992, 99993} # Cố tình gán ID sai để lấy P@5 = 0
    },
]


In [30]:
evaluation_set = [
    {
        'query': 'machine learning algorithms',
        'relevant_docs': {8964, 20472, 15168, 338, 2066}
    },
    {
        'query': 'climate change warming',
        'relevant_docs': {9209, 896, 1321, 3178}
    },
    {
        'query': 'stock market trading',
        'relevant_docs': {24446, 14408, 9}
    },
    {
        'query': 'healthy diet nutrition',
        'relevant_docs': {3701, 11426, 6418, 13001, 1041}
    },
    {
        'query': 'renewable solar energy',
        'relevant_docs': {17056, 3806, 210, 1699}
    },
    {

        'query': 'database sql query',
        'relevant_docs': {14866, 5653, 8100, 12050}
    },
]

In [26]:

def compute_precision_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    """Nhận xét & Đánh giá kết quả thực nghiệm

    Tính Precision@K:
    P@K = (# relevant documents retrieved in top K) / K
    """
    top_k_retrieved = retrieved_doc_ids[:k]
    relevant_retrieved = [doc_id for doc_id in top_k_retrieved if doc_id in relevant_doc_ids]
    return len(relevant_retrieved) / k


def compute_recall_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    """
    Tính Recall@K:
    R@K = (# relevant documents retrieved in top K) / (# relevant documents)
    """
    if not relevant_doc_ids:
        return 0.0
    top_k_retrieved = retrieved_doc_ids[:k]
    relevant_retrieved = [doc_id for doc_id in top_k_retrieved if doc_id in relevant_doc_ids]
    return len(relevant_retrieved) / len(relevant_doc_ids)


def compute_reciprocal_rank(retrieved_doc_ids, relevant_doc_ids):
    """
    Tính Reciprocal Rank (RR):
    RR = 1 / r_q nếu document relevant đầu tiên xuất hiện ở vị trí rank r_q (1-indexed).
    Nếu không tìm thấy document relevant nào trong top retrieved, trả về 0.0.
    """
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            return 1.0 / rank
    return 0.0


In [31]:
import csv
# Hàm hỗ trợ Retrieval
def get_search_results(query, vectorizer, tfidf_matrix, top_k=5):
    query_vec = vectorizer.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx = sim_scores.argsort()[::-1][:top_k]

    results = []
    for idx in top_idx:
        score = sim_scores[idx]
        if score > 0:
            results.append((int(idx), float(score)))
    return results

# ==============================================================================
# CHẠY ĐÁNH GIÁ (Sử dụng các hàm Metric bạn đã định nghĩa ở cell trước)
# ==============================================================================

k = 5
results_summary = []
csv_rows = []
all_p5 = []
all_r5 = []
all_rr = []

print("\n" + "=" * 105)
print(f"{'STT':<4} | {'Query':<32} | {'Retrieved Top 5':<28} | {'P@5':<7} | {'R@5':<7} | {'RR':<7}")
print("=" * 105)

for idx, item in enumerate(evaluation_set, start=1):
    query = item["query"]
    relevant_docs = item["relevant_docs"]

    # Thực hiện tìm kiếm (Giả sử bạn đang dùng vec_C và X_C từ Pipeline C)
    search_results = get_search_results(query, vec_C, X_C, top_k=k)
    retrieved_ids = [doc_id for doc_id, score in search_results]

    # Gọi 3 hàm tính toán bạn đã định nghĩa ở trên
    p5 = compute_precision_at_k(retrieved_ids, relevant_docs, k=k)
    r5 = compute_recall_at_k(retrieved_ids, relevant_docs, k=k)
    rr = compute_reciprocal_rank(retrieved_ids, relevant_docs)

    all_p5.append(p5)
    all_r5.append(r5)
    all_rr.append(rr)

    print(f"{idx:<4} | {query:<32} | {str(retrieved_ids):<28} | {p5:<7.4f} | {r5:<7.4f} | {rr:<7.4f}")

    # Thu thập dữ liệu chi tiết cho file CSV
    for rank, (doc_id, score) in enumerate(search_results, start=1):
        is_relevant = doc_id in relevant_docs
        preview = documents[doc_id].replace("\n", " ").strip()[:120]
        csv_rows.append({
            "Query_ID": idx,
            "Query": query,
            "P@5": f"{p5:.4f}",
            "R@5": f"{r5:.4f}",
            "RR": f"{rr:.4f}",
            "Rank": rank,
            "Doc_ID": doc_id,
            "Similarity": f"{score:.6f}",
            "Is_Relevant": is_relevant,
            "Relevant_Docs": "; ".join(map(str, sorted(list(relevant_docs)))),
            "Preview": preview
        })

# Tính Mean của các metrics
mean_p5 = sum(all_p5) / len(all_p5) if all_p5 else 0
mean_r5 = sum(all_r5) / len(all_r5) if all_r5 else 0
mrr = sum(all_rr) / len(all_rr) if all_rr else 0

print("=" * 105)
print(f"{'TRUNG BÌNH TOÀN BỘ BENCHMARK (Mean)':<69} | {mean_p5:<7.4f} | {mean_r5:<7.4f} | {mrr:<7.4f}")
print("=" * 105)

# Thêm dòng tổng kết Mean vào file CSV
csv_rows.append({
    "Query_ID": "MEAN",
    "Query": "OVERALL BENCHMARK AVERAGE",
    "P@5": f"{mean_p5:.4f}",
    "R@5": f"{mean_r5:.4f}",
    "RR": f"{mrr:.4f}",
    "Rank": "",
    "Doc_ID": "",
    "Similarity": "",
    "Is_Relevant": "",
    "Relevant_Docs": "",
    "Preview": "Mean evaluation metrics across all benchmark queries"
})

# LƯU KẾT QUẢ RA FILE .CSV
csv_filename = "evaluation_results_final.csv"
fieldnames = [
    "Query_ID", "Query", "P@5", "R@5", "RR",
    "Rank", "Doc_ID", "Similarity", "Is_Relevant",
    "Relevant_Docs", "Preview"
]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"\n[THÀNH CÔNG] Đã tự động lưu {len(csv_rows)} dòng kết quả vào file: {csv_filename}")


STT  | Query                            | Retrieved Top 5              | P@5     | R@5     | RR     
1    | machine learning algorithms      | [8964, 10909, 20472, 3298, 15168] | 0.6000  | 0.6000  | 1.0000 
2    | climate change warming           | [19142, 29504, 9209, 16935, 6575] | 0.2000  | 0.2500  | 0.3333 
3    | stock market trading             | [20511, 24446, 29636, 14408, 25545] | 0.4000  | 0.6667  | 0.5000 
4    | healthy diet nutrition           | [3701, 11426, 6418, 13001, 3180] | 0.8000  | 0.8000  | 1.0000 
5    | renewable solar energy           | [21354, 17056, 21841, 3806, 25625] | 0.4000  | 0.5000  | 0.5000 
6    | database sql query               | [14866, 11538, 5653, 25284, 27858] | 0.4000  | 0.5000  | 1.0000 
TRUNG BÌNH TOÀN BỘ BENCHMARK (Mean)                                   | 0.4667  | 0.5528  | 0.7222 

[THÀNH CÔNG] Đã tự động lưu 31 dòng kết quả vào file: evaluation_results_final.csv


In [34]:
queries_and_ids = {
    "1. machine learning algorithms": [8964, 10909, 20472, 3298, 15168],
    "2. climate change warming": [19142, 29504, 9209, 16935, 6575],
    "3. stock market trading": [20511, 24446, 29636, 14408, 25545],
    "4. healthy diet nutrition": [3701, 11426, 6418, 13001, 3180],
    "5. renewable solar energy": [21354, 17056, 21841, 3806, 25625],
    "6. database sql query": [14866, 11538, 5653, 25284, 27858]
}

print("=== NỘI DUNG PREVIEW CỦA TOÀN BỘ TOP 5 ===\n")

for query, ids in queries_and_ids.items():
    print(f"==================================================")
    print(f"QUERY: {query}")
    print(f"==================================================")
    for doc_id in ids:
        if doc_id < len(documents):
            text = documents[doc_id].replace('\n', ' ').strip()
            preview = text[:350] + " [...]" if len(text) > 350 else text
            print(f" Doc ID [{doc_id}]: {preview}\n")
    print("\n")

=== NỘI DUNG PREVIEW CỦA TOÀN BỘ TOP 5 ===

QUERY: 1. machine learning algorithms
 Doc ID [8964]: Jacob Abernethy and I have found a computationally tractable method for computing an optimal (or near optimal depending on setting) master algorithm combining expert predictions addressing this open problem. A draft is here. Some extra details: The algorithm is optimal given a small amount of side information (k in the draft). What is the best way  [...]

 Doc ID [10909]: James Neill, a Montreal-based kdb+ developer with First Derivatives, wrote a paper about how kdb+ can be used in machine learning techniques. James Neill works as a kdb+ consultant for one of the world’s largest investment banks developing a range of applications. James has also been involved in the design of training courses in data science and ma [...]

 Doc ID [20472]: It is self-evident that the content of a website plays an important role in marketing. Strategies for content marketing and SEO should be designed first

### 12. Part I — Error Analysis

Dựa trên dữ liệu văn bản thực tế, dưới đây là phân tích chi tiết cho 2 query có kết quả tốt và 2 query có kết quả kém.

#### A. Phân tích 2 Queries có kết quả tốt

**1. Query: `healthy diet nutrition` (P@5 = 0.8)**
*   **Expected relevant documents:** Giả định các tài liệu chuyên sâu về dinh dưỡng, chế độ ăn (ví dụ: 3701, 11426, 13001...).
*   **Retrieved documents:** `[3701, 11426, 6418, 13001, 3180]`
*   **Analysis:**
    1.  **Vì sao document đứng đầu (3701)?** Tài liệu này chứa các từ khóa liên quan mật thiết như "nutrition" (lặp lại 2 lần) và "health". Có thể chiều dài tổng thể của tài liệu ngắn, giúp điểm Cosine Similarity tăng cao do chuẩn hóa mẫu số nhỏ.
    2.  **Những từ nào đóng góp nhiều vào similarity?** Từ `nutrition` và `health(y)`. Đặc biệt `nutrition` là thuật ngữ đặc thù (IDF cao) giúp kéo điểm các tài liệu 3701, 11426 lên top.
    3.  **Có lexical overlap không?** Có sự trùng khớp mặt chữ rõ ràng ở các từ khóa chính.
    4.  **Có relevant document nào bị bỏ sót không?** Có (P@5 = 0.8 nghĩa là lọt 1 tài liệu có thể bị coi là rác/ít liên quan hơn).
    5.  **Failure này xuất phát từ:** **TF (Term Frequency)**. Tài liệu `6418` ("Boiled Egg Diet") lọt vào top có thể do nó lặp lại từ "Diet" quá nhiều lần (spam từ khóa), khiến thuật toán đánh giá cao dù nội dung mang tính chất quảng cáo/mẹo vặt thay vì tài liệu dinh dưỡng chuẩn mực.

**2. Query: `machine learning algorithms` (P@5 = 0.6)**
*   **Expected relevant documents:** Các tài liệu học thuật hoặc ứng dụng về thuật toán ML.
*   **Retrieved documents:** `[8964, 10909, 20472, 3298, 15168]`
*   **Analysis:**
    1.  **Vì sao document đứng đầu (8964)?** Tài liệu này chứa trực tiếp từ "learning" và lặp lại từ "algorithm" nhiều lần trong ngữ cảnh "master algorithm", "expert predictions".
    2.  **Những từ nào đóng góp nhiều vào similarity?** `learning` và `algorithm(s)`.
    3.  **Có lexical overlap không?** Có, nhưng overlap bị lệch dạng từ (Query dùng số nhiều "algorithms", Doc 8964 dùng số ít "algorithm").
    4.  **Có relevant document nào bị bỏ sót không?** Có (P@5 = 0.6 tức là trượt 2 tài liệu chuẩn khỏi top 5).
    5.  **Failure này xuất phát từ:** **Preprocessing (Stemming/Lemmatization)**. Nếu pipeline không đưa "algorithms" và "algorithm" về chung một gốc, hệ thống sẽ coi chúng là 2 chiều không gian khác nhau, làm giảm điểm Cosine trầm trọng của các tài liệu chứa từ dạng số ít.

---

#### B. Phân tích 2 Queries có kết quả kém

**3. Query: `climate change warming` (P@5 = 0.2)**
*   **Expected relevant documents:** Các bài phân tích sâu về biến đổi khí hậu (ví dụ tài liệu có chứa "greenhouse effect" như Doc 29504).
*   **Retrieved documents:** `[19142, 29504, 9209, 16935, 6575]`
*   **Analysis:**
    1.  **Vì sao document đứng đầu (19142)?** Tài liệu 19142 lặp lại cụm "climate change" (2 lần) và "global warming" (2 lần) chỉ trong một đoạn preview rất ngắn. TF của các từ này tăng đột biến.
    2.  **Những từ nào đóng góp nhiều vào similarity?** Tất cả các từ đều đóng góp do tần suất lặp lại (TF) quá dày đặc.
    3.  **Có lexical overlap không?** Trùng khớp mặt chữ 100% với mật độ cao.
    4.  **Có relevant document nào bị bỏ sót không?** Bỏ sót rất nhiều (R@5 = 0.25).
    5.  **Failure này xuất phát từ:** **TF và Lexical matching**. Thuật toán bị "đánh lừa" bởi một tài liệu nhồi nhét từ khóa (19142), đẩy các bài có chất lượng diễn đạt tốt hơn nhưng dùng từ đồng nghĩa hoặc không lặp từ nhiều lần xuống dưới.

**4. Query: `database sql query` (P@5 = 0.4)**
*   **Expected relevant documents:** Tài liệu hướng dẫn viết truy vấn cơ sở dữ liệu.
*   **Retrieved documents:** `[14866, 11538, 5653, 25284, 27858]`
*   **Analysis:**
    1.  **Vì sao document đứng đầu (14866)?** Tài liệu này chứa "SQL Database", "database", "databases" lặp lại liên tục.
    2.  **Những từ nào đóng góp nhiều vào similarity?** `database` và `sql` đóng góp gần như toàn bộ điểm số. Từ `query` dường như vắng mặt trong các tài liệu top đầu.
    3.  **Có lexical overlap không?** Chỉ trùng khớp một phần (partial overlap: có "database sql" nhưng thiếu "query").
    4.  **Có relevant document nào bị bỏ sót không?** Có (R@5 = 0.5).
    5.  **Failure này xuất phát từ:** **IDF và Lexical matching**. Các từ "database" và "SQL" có thể xuất hiện cùng nhau rất nhiều, tạo ra điểm TF-IDF lớn, làm lu mờ hoàn toàn trọng số của từ "query" nếu nó không xuất hiện trực tiếp.

---

#### C. Failure case quan trọng nhất

**Sự bất lực của Lexical Matching trước Ngữ Nghĩa (Semantic Gap)**

Dựa vào Query 6 (`database sql query`), ta thấy rõ giới hạn chí mạng của CountVectorizer/TF-IDF: **Chỉ đếm mặt chữ, không hiểu ý nghĩa.**

**Giải thích kỹ:**
Khi người dùng tìm kiếm cụm `"database sql query"`, hệ thống trả về Doc 11538 (nói về *"SQL commands"*) và Doc 25284 (nói về *"stored procedure"*, *"SqlCommand"*, *"recordset"*).

Về mặt ngữ nghĩa chuyên ngành, "SQL commands" hay "stored procedure" chính là các hình thức thực thi của "SQL query". Một con người đọc vào sẽ đánh giá các tài liệu này cực kỳ relevant. Tuy nhiên, hệ thống TF-IDF lại đánh giá rất thấp (hoặc cho điểm 0 đối với chiều dữ liệu của từ `query`), bởi vì từ `query` **không xuất hiện chính xác từng chữ cái một** trong văn bản.

Trong không gian Vector (VSM), trục tọa độ của từ `query` vuông góc hoàn toàn với trục tọa độ của từ `commands`. Phép nhân Cosine Similarity giữa chúng bằng 0. Lỗi này (Failure) xuất phát thuần túy từ **Lexical matching**. Để giải quyết triệt để, hệ thống bắt buộc phải nâng cấp lên các phương pháp Semantic Search (sử dụng Word Embeddings như Word2Vec, BERT) để ánh xạ các từ đồng nghĩa và ngữ cảnh vào chung một không gian vector.

# 13. Part J — From Failure to the Next NLP Representation

## 1. Hạn chế cốt lõi của mô hình TF-IDF

Mô hình TF-IDF và không gian vector truyền thống (Vector Space Model) biểu diễn văn bản dựa trên **thống kê từ vựng bề mặt (Lexical Statistics)**. Cách tiếp cận này bộc lộ những điểm yếu chí mạng:

1. **Giả định tính trực giao (Orthogonality Assumption)**:
   - Mỗi từ trong từ điển $V$ được coi là một chiều độc lập trong không gian $|V|$ chiều.
   - Khoảng cách giữa hai từ bất kỳ (dù đồng nghĩa như `"heart attack"` và `"myocardial infarction"`, hay không liên quan như `"cat"` và `"refrigerator"`) đều như nhau: tích vô hướng bằng 0.
2. **Vấn đề từ đồng nghĩa (Synonymy / Vocabulary Mismatch)**:
   - Người dùng và tác giả văn bản thường sử dụng các từ vựng khác nhau để diễn đạt cùng một khái niệm. TF-IDF không có khả năng nhận ra sự tương đồng này nếu không trùng khớp chính xác mặt chữ.
3. **Vấn đề từ đa nghĩa (Polysemy) và Mất ngữ cảnh**:
   - Từ `"bank"` trong *"river bank"* (bờ sông) và *"bank account"* (tài khoản ngân hàng) đều có cùng một tọa độ và cùng một giá trị IDF.
   - Bỏ qua trật tự từ (Bag-of-Words): câu *"dog bites man"* và *"man bites dog"* có biểu diễn vector hoàn toàn giống nhau dù ý nghĩa trái ngược.

---

## 2. Câu hỏi cuối buổi: Biểu diễn Similarity về Nghĩa thay vì Similarity về Từ

> **Câu hỏi**: *Làm thế nào để biểu diễn được similarity về nghĩa (semantic similarity) thay vì chỉ similarity về từ (lexical similarity)?*

**Trả lời**:
- Các từ có ngữ nghĩa tương đồng (như *"heart attack"* và *"infarction"*) thường xuất hiện chung với những từ ngữ cảnh tương tự (như *"hospital"*, *"patient"*, *"doctor"*, *"chest pain"*, *"treatment"*).
- Thay vì biểu diễn từ dưới dạng vector thưa one-hot trực giao, ta ánh xạ các từ vào một **không gian vector liên tục, dày đặc (Dense Vector Space $\mathbb{R}^d$ với $d \ll |V|$, thường từ 100 đến 1024 chiều)**.
- Trong không gian này, các từ hoặc văn bản có sự tương đồng về ngữ cảnh sẽ có vị trí hình học gần nhau, thể hiện qua điểm **Cosine Similarity cao** ngay cả khi không trùng lặp mặt chữ.
